**Prepare the Name-Search Evaluation Set**

This notebook builds the evaluation set on which the name-search experiments run: the first half of the preparation stage of the experiment design (`./docs/experiment_design.pdf`, §2.1), as the manifest (`./docs/MANIFEST.md`, §4.1–4.4) specifies it; `2_generate_queries.ipynb` completes it with the query sets. It reads the Gemini transcription `../ocr_pipeline/extracted_text.csv`, which it never modifies, and the hand-built given-name dictionary `./dictionaries/d1_curated.csv`. It normalises every name with the rules the production import will use, merges rows that describe the same person, and draws a sealed test split and five development folds. The test persons go to their own file, which the development notebooks never open. It outputs:

1. `./eval/profile.json`: the data profile of the current transcription, recomputing the figures the design quotes from the older extraction, including the relation between each person's Spanish and Hebrew given names (the design's Figure 1)
2. `./eval/persons_dev.csv` and `./eval/persons_test.csv`: one row per target person, with its graves, the rows merged into it and its normalised name tokens
3. `./eval/splits.csv`: each person's split (`dev` or `test`) and development fold (1–5)
4. `./eval/d1_forms.csv`: the forms of the curated dictionary D1 with their normalised spellings
5. `./review/d1_review.csv`: the dictionary, left for human review

**1. Configuration and Helpers**

This cell defines the settings and the normalisation that every later cell reuses. The normalisation is written as one function per script so that the production `import.py` can reuse it unchanged.

1. `SEED`: the seed of every random step, 333 as the manifest requires
2. `normalise_latin` and `normalise_hebrew`: the manifest's §4.2 rules. Latin text is decomposed, stripped of accents, lower-cased and reduced to letters. Hebrew text loses its points and cantillation, its final letters become medial forms, the Yiddish ligatures become letter pairs, and geresh, gershayim and quotation marks are removed
3. `display_form`: the spelling an embedding model sees. It keeps accents and Hebrew final letters, and converts a name carved in capitals (`REBECA`) to title case (`Rebeca`), the casing in which a visitor types it
4. `name_tokens`: splits a name field into `NameToken`s, dropping single-letter initials and, in surnames, the married-name particle (`de`, `דע`)

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  shared by every later cell
#
# A name field becomes a list of tokens, each with a normalised form (what
# exact and lexical matching compare) and a display form (what an embedding
# model reads). Script is detected per token, not per column, so a Latin
# word stray in a Hebrew column is still treated as Latin:
#   "S. de Blejer"      → [blejer]
#   "Sabbah de Cohen"   → [sabbah, cohen]
#   "חיה שרה"            → [חיה, שרה]
# Every random step is seeded with SEED, so a re-run writes identical files.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import abydos.phonetic as bm_phonetic
import numpy as np
from abydos.phonetic._beider_morse_data import BMDATA
from anyascii import anyascii
from safetensors.numpy import load_file
from sklearn.model_selection import StratifiedGroupKFold

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
SOURCE_CSV      = Path("../ocr_pipeline/extracted_text.csv")   # read only
D1_SOURCE       = Path("dictionaries/d1_curated.csv")          # hand-built D1
PREFLIGHT_FILE  = Path("results/preflight.json")               # asset statuses
EVAL_DIR        = Path("eval")                                 # generated inputs
REVIEW_DIR      = Path("review")                               # human-review files
SEED            = 333          # every random step
MIN_CONFIDENCE  = 85           # eligibility: extraction confidence, 0–100
TEST_SHARE_FOLD = 5            # 1 of 5 grouped folds (20 %) becomes the test split
DEV_FOLDS       = 5            # development folds
LINK_COSINE     = 0.85         # Figure 1: encoder cosine that counts as linked
RULE            = "─" * 72     # frames every printed report

# The transcription's name columns, by field and script of the column.
NAME_COLUMNS = {
    ("given", "latin"): "given_name_es", ("given", "hebrew"): "given_name_he",
    ("surname", "latin"): "surname_es", ("surname", "hebrew"): "surname_he",
}
SURNAME_PARTICLES = {"de", "דע"}                       # dropped from surnames
FINAL_TO_MEDIAL   = str.maketrans("ךםןףץ", "כמנפצ")     # final letters → medial forms
MAQAF             = "\N{HEBREW PUNCTUATION MAQAF}"     # the Hebrew hyphen
HEBREW_LETTERS    = "\N{HEBREW LETTER ALEF}-\N{HEBREW LETTER TAV}"            # a regex range
HEBREW_MARKS      = re.compile("[\N{HEBREW ACCENT ETNAHTA}-\N{HEBREW POINT QAMATS QATAN}]")
HEBREW_NON_LETTER = re.compile(f"[^{HEBREW_LETTERS} ]")  # anything but a letter or space
TOKEN_SPLIT       = re.compile(rf"[\s\-{MAQAF}]+")       # whitespace, hyphen, maqaf
YIDDISH_LIGATURES = {"\N{HEBREW LIGATURE YIDDISH DOUBLE VAV}": "וו",
                     "\N{HEBREW LIGATURE YIDDISH VAV YOD}": "וי",
                     "\N{HEBREW LIGATURE YIDDISH DOUBLE YOD}": "יי"}


@dataclass(frozen=True)
class NameToken:
    """One token of a person's name: its `field` (given or surname), source
    `column`, `script` (latin or hebrew), normalised form `norm`, compared by
    exact and lexical matching, and `display` spelling, read by embeddings."""

    field: str
    column: str
    script: str
    norm: str
    display: str


def token_script(text: str) -> str:
    """Return `hebrew` if `text` holds any Hebrew character, else `latin`."""
    return "hebrew" if any(0x0590 <= ord(char) <= 0x05FF for char in text) else "latin"


def normalise_latin(text: str) -> str:
    """Normalise Latin-script text: no accents, lower case, letters only."""
    decomposed = unicodedata.normalize("NFKD", text.replace("-", " "))
    letters = "".join(char for char in decomposed
                      if not unicodedata.combining(char) and (char.isalpha() or char.isspace()))
    return " ".join(letters.lower().split())


def normalise_hebrew(text: str) -> str:
    """Normalise Hebrew-script text: no points, medial letters, letters only.

    NFKD first splits presentation forms such as `וּ` (one code point) into
    the letter and its point, so that removing the point keeps the letter.
    """
    text = unicodedata.normalize("NFKD", text).replace(MAQAF, " ").replace("-", " ")
    text = HEBREW_MARKS.sub("", text)
    for ligature, letters in YIDDISH_LIGATURES.items():
        text = text.replace(ligature, letters)
    text = HEBREW_NON_LETTER.sub("", text.translate(FINAL_TO_MEDIAL))
    return " ".join(text.split())


def display_form(piece: str, script: str) -> str:
    """Return the spelling of one recorded token that an embedding model reads.

    Hebrew keeps its final letters but loses points, ligatures and quotation
    marks; Latin keeps its accents, and a token carved in capitals is turned
    into title case.
    """
    piece = unicodedata.normalize("NFC", piece)
    if script == "hebrew":
        piece = HEBREW_MARKS.sub("", piece)
        for ligature, letters in YIDDISH_LIGATURES.items():
            piece = piece.replace(ligature, letters)
        return re.sub(f"[^{HEBREW_LETTERS}]", "", piece)
    piece = "".join(char for char in piece if char.isalpha() or char == "'")
    return piece.capitalize() if piece.isupper() else piece


def name_tokens(raw: str, field_name: str, column: str) -> list[NameToken]:
    """Split one name field into normalised tokens.

    Args:
        raw: the field as transcribed, e.g. `S. de Blejer`.
        field_name: `given` or `surname`; only surnames drop the particle.
        column: the transcription column, kept on every token.

    Returns:
        The tokens in field order, without initials (single letters) and,
        for surnames, without `de` and `דע`.
    """
    tokens = []
    for piece in TOKEN_SPLIT.split(raw.strip()):
        script = token_script(piece)
        norm = (normalise_hebrew(piece) if script == "hebrew" else normalise_latin(piece))
        norm = norm.replace(" ", "")
        if len(norm) < 2 or (field_name == "surname" and norm in SURNAME_PARTICLES):
            continue
        tokens.append(NameToken(field_name, column, script, norm, display_form(piece, script)))
    return tokens


def row_tokens(row: dict) -> list[NameToken]:
    """Return every name token of one transcription row, column by column."""
    return [token for (field_name, _), column in NAME_COLUMNS.items()
            for token in name_tokens(row.get(column) or "", field_name, column)]


def has_tokens(tokens: list[NameToken], field_name: str, script: str) -> bool:
    """Return True if `tokens` include one of `field_name` in `script`."""
    return any(t.field == field_name and t.script == script for t in tokens)


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


source_rows = read_csv(SOURCE_CSV)
preflight = json.loads(PREFLIGHT_FILE.read_text(encoding="utf-8"))
print(f"✓ Configuration and helpers loaded ({len(source_rows)} transcription rows, "
      f"seed {SEED}).")

✓ Configuration and helpers loaded (825 transcription rows, seed 333).


**2. Data Profile**

This cell takes `source_rows` and recomputes, on the current transcription, every count the design quotes from the older extraction (828 rows, 668 with a given name and a surname, 657 of them eligible). A row is eligible when it records a given name and a surname, in either script, and its extraction confidence is at least `MIN_CONFIDENCE`. It produces:

1. `eligible_rows`: the indices of the eligible rows in `source_rows`
2. `profile`: the counts by section, by script coverage, of married-name particles in the women's Latin surnames, of Yiddish orthography (`ע`, `טש`) in Hebrew surnames, and of double Hebrew given names

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Data profile  —  the design's §2 figures, recomputed on the current file
#
# "Records a given name and a surname" is judged on the tokens left after
# normalisation, so a field holding only an initial does not count.
# ─────────────────────────────────────────────────────────────────────────────
if "source_rows" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def full_name_scripts(tokens: list[NameToken]) -> set[str]:
    """Return the scripts in which `tokens` hold both a given name and a surname."""
    return {script for script in ("latin", "hebrew")
            if has_tokens(tokens, "given", script) and has_tokens(tokens, "surname", script)}


def is_named(tokens: list[NameToken]) -> bool:
    """Return True if `tokens` hold a given name and a surname, in any scripts."""
    return (any(t.field == "given" for t in tokens)
            and any(t.field == "surname" for t in tokens))


source_tokens = [row_tokens(row) for row in source_rows]
named_rows = [index for index, tokens in enumerate(source_tokens) if is_named(tokens)]
eligible_rows = [index for index in named_rows
                 if int(source_rows[index]["confidence"] or 0) >= MIN_CONFIDENCE]

_women_latin_surnames = [row["surname_es"] for row in source_rows
                         if row["section"] == "women" and row["surname_es"].strip()]
_hebrew_surnames = [row["surname_he"] for row in source_rows if row["surname_he"].strip()]
_hebrew_given = [tokens for tokens in source_tokens if has_tokens(tokens, "given", "hebrew")]
profile = {
    "source": str(SOURCE_CSV),
    "rows": len(source_rows),
    "graves": len({(row["section"], row["row"], row["stone"]) for row in source_rows}),
    "rows_by_section": dict(Counter(row["section"] for row in source_rows)),
    "rows_with_given_and_surname": len(named_rows),
    "eligible_rows": len(eligible_rows),
    "named_rows_full_name_latin": sum("latin" in full_name_scripts(source_tokens[i])
                                      for i in named_rows),
    "named_rows_full_name_hebrew": sum("hebrew" in full_name_scripts(source_tokens[i])
                                       for i in named_rows),
    "named_rows_full_name_both": sum(len(full_name_scripts(source_tokens[i])) == 2
                                     for i in named_rows),
    "women_latin_surnames": len(_women_latin_surnames),
    "women_latin_surnames_with_de": sum(" de " in f" {normalise_latin(name)} "
                                        for name in _women_latin_surnames),
    "hebrew_surnames": len(_hebrew_surnames),
    "hebrew_surnames_yiddish_orthography": sum("ע" in name or "טש" in name
                                               for name in _hebrew_surnames),
    "hebrew_given_names": len(_hebrew_given),
    "hebrew_given_names_double": sum(sum(t.field == "given" and t.script == "hebrew"
                                         for t in tokens) > 1 for tokens in _hebrew_given),
}

print(f"{RULE}\nData profile: {SOURCE_CSV}\n{RULE}")
for _key, _value in profile.items():
    if _key != "source":
        print(f"  {_key:<38} {_value}")
print(f"{RULE}\n  {len(eligible_rows)} of {len(source_rows)} rows eligible "
      f"(given name and surname, confidence ≥ {MIN_CONFIDENCE})")

────────────────────────────────────────────────────────────────────────
Data profile: ../ocr_pipeline/extracted_text.csv
────────────────────────────────────────────────────────────────────────
  rows                                   825
  graves                                 794
  rows_by_section                        {'children': 198, 'women': 627}
  rows_with_given_and_surname            655
  eligible_rows                          639
  named_rows_full_name_latin             577
  named_rows_full_name_hebrew            441
  named_rows_full_name_both              368
  women_latin_surnames                   513
  women_latin_surnames_with_de           268
  hebrew_surnames                        447
  hebrew_surnames_yiddish_orthography    166
  hebrew_given_names                     579
  hebrew_given_names_double              147
────────────────────────────────────────────────────────────────────────
  639 of 825 rows eligible (given name and surname, confidence ≥ 85)


**3. Given-Name Relation (Figure 1)**

This cell takes `source_rows` and repeats the design's Figure 1 on the current file: for every row that records a given name in both scripts, it classifies the relation between the first Spanish given-name token and the Hebrew given-name tokens with the design's rule.

1. **Phonetically linked**: some Hebrew token shares a Daitch–Mokotoff code with the Spanish token after AnyAscii romanisation, shares a Beider–Morse token (approximate, or Hebrew-value pass), or reaches an E1 encoder cosine of at least `LINK_COSINE`
2. **Shared initial sound only**: otherwise, some pair of their Daitch–Mokotoff codes starts with the same digit
3. **Neither**: the rest

The Daitch–Mokotoff codes here come from Abydos, since this notebook runs no database; the experiments compute L3 in PostgreSQL. It adds `given_name_relation` to `profile`, with three examples of each class.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Given-name relation  —  the design's Figure 1 on the current transcription
#
# The classification sizes the cross-script problem; it is approximate, as
# the design says. Beider–Morse runs with the Ashkenazic rules: Latin input
# with a fixed language set (automatic detection takes "Svarc" for
# Hungarian), Hebrew input with automatic detection, which finds Hebrew.
# ─────────────────────────────────────────────────────────────────────────────
if "profile" not in globals():
    raise RuntimeError("Run the 'Data Profile' cell first.")

E1_CHECKPOINT  = Path(preflight["E1"].get("checkpoint", "assets/missing"))
LATIN_LANGUAGES = "spanish,german,polish,russian,english"   # Beider–Morse, Latin input


def layer_norm(x: np.ndarray, gain: np.ndarray, bias: np.ndarray) -> np.ndarray:
    """Normalise the last axis of `x`, then scale by `gain` and shift by `bias`."""
    mean = x.mean(-1, keepdims=True)
    return (x - mean) / np.sqrt(((x - mean) ** 2).mean(-1, keepdims=True) + 1e-5) * gain + bias


def e1_attention(w: dict, x: np.ndarray, prefix: str) -> np.ndarray:
    """Return the 8-head self-attention update of E1 layer `prefix` for the states `x`."""
    heads, head_dim = 8, x.shape[1] // 8
    h = layer_norm(x, w[prefix + "norm1.weight"], w[prefix + "norm1.bias"])
    qkv = h @ w[prefix + "self_attn.in_proj_weight"].T + w[prefix + "self_attn.in_proj_bias"]
    q, k, v = (t.reshape(len(x), heads, head_dim).transpose(1, 0, 2) for t in np.split(qkv, 3, -1))
    scores = np.exp(q @ k.transpose(0, 2, 1) / np.sqrt(head_dim))
    attended = ((scores / scores.sum(-1, keepdims=True)) @ v).transpose(1, 0, 2).reshape(x.shape)
    out_proj = prefix + "self_attn.out_proj."
    return attended @ w[out_proj + "weight"].T + w[out_proj + "bias"]


def e1_feed_forward(w: dict, x: np.ndarray, prefix: str) -> np.ndarray:
    """Return the ReLU feed-forward update of E1 layer `prefix` for the states `x`."""
    h = layer_norm(x, w[prefix + "norm2.weight"], w[prefix + "norm2.bias"])
    hidden = np.maximum(0, h @ w[prefix + "linear1.weight"].T + w[prefix + "linear1.bias"])
    return hidden @ w[prefix + "linear2.weight"].T + w[prefix + "linear2.bias"]


def e1_encode(w: dict, text: str) -> np.ndarray:
    """Return E1's unit-length embedding of `text`: the NumPy forward pass of the
    manifest's Appendix A, over the UTF-8 bytes, mean-pooled and normalised."""
    ids = list(text.encode("utf-8"))[:256]
    x = w["embedding.weight"][ids] + w["pos_embedding.weight"][:len(ids)]
    for layer in range(6):
        x = x + e1_attention(w, x, f"transformer.layers.{layer}.")
        x = x + e1_feed_forward(w, x, f"transformer.layers.{layer}.")
    pooled = x.mean(0)
    return pooled / np.linalg.norm(pooled)


def beider_morse_encoders() -> dict[str, object]:
    """Return the four Beider–Morse encoders, keyed `<script>_<pass>`.

    The Hebrew-value pass is enabled as in the manifest's Appendix B: the
    empty language-specific Hebrew rule sets are filled in, and the private
    match mode is set to `hebrew`.
    """
    hebrew_rules = BMDATA["ash"]["hebrew"]
    for language in BMDATA["ash"]["approx"]:
        if language != "common" and language not in hebrew_rules:
            hebrew_rules[language] = ()
    encoders = {}
    for script, language in (("latin", LATIN_LANGUAGES), ("hebrew", 0)):
        for match_mode in ("approx", "hebrew"):
            encoder = bm_phonetic.BeiderMorse(name_mode="ash", language_arg=language)
            encoder._match_mode = match_mode  # pylint: disable=protected-access
            encoders[f"{script}_{match_mode}"] = encoder
    return encoders


BEIDER_MORSE = beider_morse_encoders()
DAITCH_MOKOTOFF = bm_phonetic.DaitchMokotoff()


def dm_codes(token: NameToken) -> set[str]:
    """Return the Daitch–Mokotoff codes of a token, romanising Hebrew first."""
    text = anyascii(token.norm) if token.script == "hebrew" else token.norm
    return set(DAITCH_MOKOTOFF.encode(text)) - {"000000"}


def bm_agree(first: NameToken, second: NameToken) -> bool:
    """Return True if two tokens share a Beider–Morse token in either pass."""
    for match_mode in ("approx", "hebrew"):
        tokens_first = BEIDER_MORSE[f"{first.script}_{match_mode}"].encode(first.norm).split()
        tokens_second = BEIDER_MORSE[f"{second.script}_{match_mode}"].encode(second.norm).split()
        if set(tokens_first) & set(tokens_second):
            return True
    return False


def given_name_relation(spanish: NameToken, hebrew: list[NameToken], e1: dict | None) -> str:
    """Classify one row's Spanish given name against its Hebrew given names.

    Returns:
        `phonetic`, `initial` or `neither`, by the design's Figure 1 rule.
    """
    spanish_codes = dm_codes(spanish)
    for token in hebrew:
        cosine = (float(e1_encode(e1, spanish.display) @ e1_encode(e1, token.display))
                  if e1 else 0.0)
        if spanish_codes & dm_codes(token) or bm_agree(spanish, token) or cosine >= LINK_COSINE:
            return "phonetic"
    spanish_initials = {code[0] for code in spanish_codes}
    if any(code[0] in spanish_initials for token in hebrew for code in dm_codes(token)):
        return "initial"
    return "neither"


_encoder = load_file(str(E1_CHECKPOINT)) if preflight["E1"]["status"] == "ok" else None
_relations, _examples = Counter(), defaultdict(list)
for _tokens in source_tokens:
    _spanish = [t for t in _tokens if t.field == "given" and t.script == "latin"]
    _hebrew = [t for t in _tokens if t.field == "given" and t.script == "hebrew"]
    if _spanish and _hebrew:
        _relation = given_name_relation(_spanish[0], _hebrew, _encoder)
        _relations[_relation] += 1
        if len(_examples[_relation]) < 3:
            _examples[_relation].append(
                f"{_spanish[0].display}–{' '.join(t.display for t in _hebrew)}")
profile["given_name_relation"] = {
    "rows": sum(_relations.values()), "encoder_used": _encoder is not None,
    **{relation: _relations[relation] for relation in ("phonetic", "initial", "neither")},
    "examples": dict(_examples)}

print(f"{RULE}\nGiven-name relation: rows with a given name in both scripts\n{RULE}")
for _relation in ("phonetic", "initial", "neither"):
    print(f"  {_relation:<10} {_relations[_relation]:>4}  "
          f"({_relations[_relation] / max(1, sum(_relations.values())):5.1%})  "
          f"e.g. {', '.join(_examples[_relation])}")
print(f"{RULE}\n  {sum(_relations.values())} rows classified "
      f"({'with' if _encoder else 'without'} the E1 encoder)")

────────────────────────────────────────────────────────────────────────
Given-name relation: rows with a given name in both scripts
────────────────────────────────────────────────────────────────────────
  phonetic    187  (39.9%)  e.g. Samuel–שמואל, Daniel–דניעל, Babe–באבע
  initial     186  (39.7%)  e.g. Raquel–רחל, Valentin–אב, Jose–יוסף
  neither      96  (20.5%)  e.g. Hector–מרדכי אריה, Elena–יעכט מלכה, Bernardo–דובעריש
────────────────────────────────────────────────────────────────────────
  469 rows classified (with the E1 encoder)


**4. Target Persons**

This cell takes `eligible_rows` and merges the rows that describe the same person on two stones into one **target person**, as the design's §2.1 prescribes. Two rows are merged when either key agrees and is complete: the normalised Latin given name, last Latin surname token and death year, or the normalised Hebrew given name, Hebrew surname and death year. Merging is transitive. Each person keeps every grave as a correct answer and the union of its rows' name tokens. It produces:

1. `persons`: one `Person` per target, numbered `P0001`, `P0002`, ... in section, row and stone order
2. each person's script coverage (`latin`, `hebrew` or `both`, from its tokens), lowest extraction confidence and **family key**: the last Latin surname token, the husband's surname in "X de Y", else the last Hebrew surname token

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Target persons  —  duplicate rows merged with a union–find
#
# The Serebrinsky brothers on stones 7 and 8 of the children's section are
# the design's example: the same names and year of death on two stones.
# A person's confidence is the lowest of its rows, so a breakdown by
# confidence band never credits a merged person with its better reading.
# ─────────────────────────────────────────────────────────────────────────────
if "eligible_rows" not in globals():
    raise RuntimeError("Run the 'Data Profile' cell first.")


@dataclass
class Person:
    """One target person of the evaluation set.

    Attributes:
        person_id: `P0001`, `P0002`, ... in section, row and stone order.
        section: `women` or `children`.
        rows: the indices in `source_rows` of the rows merged into it.
        tokens: the union of the rows' name tokens, without repeats.
        split: `dev` or `test`, set by the 'Splits' cell.
        fold: the development fold, 1–5, or 0 for a test person.
    """

    person_id: str
    section: str
    rows: list[int]
    tokens: list[NameToken] = field(default_factory=list)
    split: str = ""
    fold: int = 0

    @property
    def graves(self) -> list[str]:
        """Return the person's graves as `section/row/stone`."""
        return sorted({"/".join(source_rows[i][key] for key in ("section", "row", "stone"))
                       for i in self.rows})

    @property
    def coverage(self) -> str:
        """Return `latin`, `hebrew` or `both`: the scripts of its tokens."""
        scripts = {token.script for token in self.tokens}
        return "both" if len(scripts) == 2 else scripts.pop()

    @property
    def confidence(self) -> int:
        """Return the lowest extraction confidence of its rows."""
        return min(int(source_rows[i]["confidence"]) for i in self.rows)

    @property
    def family_key(self) -> str:
        """Return the last Latin surname token, else the last Hebrew one."""
        for script in ("latin", "hebrew"):
            surnames = [t.norm for t in self.tokens if t.field == "surname" and t.script == script]
            if surnames:
                return surnames[-1]
        return ""


def merge_keys(index: int) -> list[tuple]:
    """Return the complete duplicate keys of one eligible row."""
    tokens, year = source_tokens[index], source_rows[index]["death_year"].strip()
    keys = []
    for script in ("latin", "hebrew"):
        given = " ".join(t.norm for t in tokens if t.field == "given" and t.script == script)
        surnames = [t.norm for t in tokens if t.field == "surname" and t.script == script]
        surname = surnames[-1] if script == "latin" and surnames else " ".join(surnames)
        if given and surname and year:
            keys.append((script, given, surname, year))
    return keys


def merge_rows(indices: list[int]) -> list[list[int]]:
    """Group row indices that share a duplicate key, transitively.

    Returns:
        The groups, each sorted, in order of their first row.
    """
    parent = {index: index for index in indices}

    def root(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    first_with_key = {}
    for index in indices:
        for key in merge_keys(index):
            if key in first_with_key:
                parent[root(index)] = root(first_with_key[key])
            else:
                first_with_key[key] = index
    groups = defaultdict(list)
    for index in indices:
        groups[root(index)].append(index)
    return sorted(groups.values(), key=lambda group: group[0])


def unique_tokens(indices: list[int]) -> list[NameToken]:
    """Return the union of the rows' tokens, keeping each (field, column, norm) once."""
    seen, tokens = set(), []
    for index in indices:
        for token in source_tokens[index]:
            if (token.field, token.column, token.norm) not in seen:
                seen.add((token.field, token.column, token.norm))
                tokens.append(token)
    return tokens


persons = [Person(f"P{number:04d}", source_rows[group[0]]["section"], group, unique_tokens(group))
           for number, group in enumerate(merge_rows(eligible_rows), start=1)]
_merged = [person for person in persons if len(person.rows) > 1]
profile["target_persons"] = len(persons)
profile["merged_persons"] = len(_merged)
profile["persons_by_coverage"] = dict(Counter(person.coverage for person in persons))

print(f"{RULE}\nTarget persons: {len(eligible_rows)} eligible rows\n{RULE}")
for _coverage, _count in sorted(profile["persons_by_coverage"].items()):
    print(f"  {_coverage:<8} {_count:>4} persons")
for _person in _merged[:5]:
    print(f"  merged   {_person.person_id}: {', '.join(_person.graves)}")
print(f"{RULE}\n  {len(persons)} target persons, {len(_merged)} of them merged from "
      f"two or more rows")

────────────────────────────────────────────────────────────────────────
Target persons: 639 eligible rows
────────────────────────────────────────────────────────────────────────
  both      467 persons
  hebrew     63 persons
  latin     107 persons
  merged   P0005: children/0/7, children/0/8
  merged   P0006: children/0/7, children/0/8
────────────────────────────────────────────────────────────────────────
  637 target persons, 2 of them merged from two or more rows


**5. Splits**

This cell takes `persons` and draws the **test split**, a fifth of the persons that only Experiment 4 may read, and divides the rest into five **development folds**. Both draws use stratified group k-fold with `SEED`: strata are section × script coverage, and groups are family keys, so that every person of one family surname falls in the same split and fold and no surname appears in both a dictionary's training folds and its evaluation fold. It sets each person's `split` and `fold` and writes:

1. `./eval/splits.csv`: `person_id, split, fold`
2. `./eval/persons_dev.csv` and `./eval/persons_test.csv`: each person's section, graves, merged rows, confidence, coverage, family key and tokens (as JSON)

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Splits  —  a sealed 20 % test split and five development folds
#
# StratifiedGroupKFold keeps groups whole while matching the strata's
# proportions as closely as whole groups allow. The first of five grouped
# folds is the test split; the other four fifths are split again in five.
# ─────────────────────────────────────────────────────────────────────────────
if "persons" not in globals():
    raise RuntimeError("Run the 'Target Persons' cell first.")


def grouped_folds(members: list[Person], n_folds: int) -> list[int]:
    """Assign each person a fold, 1 to `n_folds`, keeping family keys together.

    Returns:
        The fold of each person of `members`, in order.
    """
    strata = [f"{person.section}|{person.coverage}" for person in members]
    groups = [person.family_key for person in members]
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    folds = [0] * len(members)
    for fold, (_, held_out) in enumerate(splitter.split(strata, strata, groups), start=1):
        for index in held_out:
            folds[index] = fold
    return folds


def person_record(person: Person) -> dict:
    """Return the CSV row of one person."""
    return {
        "person_id": person.person_id, "section": person.section, "split": person.split,
        "fold": person.fold or "", "graves": json.dumps(person.graves),
        "merged_from": json.dumps(person.rows), "confidence": person.confidence,
        "coverage": person.coverage, "family_key": person.family_key,
        "death_year": source_rows[person.rows[0]]["death_year"],
        "tokens": json.dumps([vars(token) for token in person.tokens], ensure_ascii=False),
    }


for _person, _fold in zip(persons, grouped_folds(persons, TEST_SHARE_FOLD)):
    _person.split = "test" if _fold == 1 else "dev"
dev_persons = [person for person in persons if person.split == "dev"]
test_persons = [person for person in persons if person.split == "test"]
for _person, _fold in zip(dev_persons, grouped_folds(dev_persons, DEV_FOLDS)):
    _person.fold = _fold
persons_by_id = {person.person_id: person for person in persons}

EVAL_DIR.mkdir(exist_ok=True)
write_csv(EVAL_DIR / "splits.csv", [{"person_id": p.person_id, "split": p.split,
                                     "fold": p.fold or ""} for p in persons])
write_csv(EVAL_DIR / "persons_dev.csv", [person_record(p) for p in dev_persons])
write_csv(EVAL_DIR / "persons_test.csv", [person_record(p) for p in test_persons])
profile["split_sizes"] = {"dev": len(dev_persons), "test": len(test_persons),
                          **{f"fold_{k}": sum(p.fold == k for p in dev_persons)
                             for k in range(1, DEV_FOLDS + 1)}}

print(f"{RULE}\nSplits: section × coverage strata, family-key groups\n{RULE}")
for _stratum in sorted({f"{p.section}|{p.coverage}" for p in persons}):
    _members = [p for p in persons if f"{p.section}|{p.coverage}" == _stratum]
    _counts = [sum(p.split == "test" for p in _members)]
    _counts += [sum(p.fold == k for p in _members) for k in range(1, DEV_FOLDS + 1)]
    print(f"  {_stratum:<16} test {_counts[0]:>3}   folds "
          + " ".join(f"{c:>3}" for c in _counts[1:]))
print(f"{RULE}\n  {len(dev_persons)} development persons in {DEV_FOLDS} folds, "
      f"{len(test_persons)} test persons")

────────────────────────────────────────────────────────────────────────
Splits: section × coverage strata, family-key groups
────────────────────────────────────────────────────────────────────────
  children|both    test  10   folds   7   7   8   7   7
  children|hebrew  test   2   folds   1   1   1   1   2
  children|latin   test   4   folds   3   3   4   4   3
  women|both       test  85   folds  67  68  67  67  67
  women|hebrew     test  11   folds   9   9   8   9   9
  women|latin      test  18   folds  13  13  14  14  14
────────────────────────────────────────────────────────────────────────
  507 development persons in 5 folds, 130 test persons


**6. Curated Dictionary D1**

This cell takes `D1_SOURCE`, the given-name equivalence dictionary compiled by hand from general knowledge of Hebrew, Yiddish and Spanish names, and prepares it for the experiments. It checks that every row has a known script and kind, that the script matches the form's characters, and that no normalised form belongs to two groups, which would silently merge the groups in D3. It writes:

1. `./eval/d1_forms.csv`: each form with its group, script, kind, normalised form and display form
2. `./review/d1_review.csv`: a copy of the dictionary with an empty `reviewer_ok` column, left for human review

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Curated dictionary D1  —  validated, normalised and copied for review
#
# Kinds: cognate (the same name), yiddish (a Yiddish form), translation (a
# name of the same meaning: Zeev/Wolf, Aryeh/Leib), hypocoristic (a
# diminutive) and civil (a civil name conventionally given for the Hebrew
# or Yiddish one: Jaime for Chaim). `civil` extends the manifest's four kinds.
# ─────────────────────────────────────────────────────────────────────────────
if "persons" not in globals():
    raise RuntimeError("Run the 'Target Persons' cell first.")

D1_KINDS = {"cognate", "yiddish", "translation", "hypocoristic", "civil"}


def d1_problems(rows: list[dict]) -> list[str]:
    """Return every problem found in the rows of the curated dictionary."""
    problems, groups_of_form = [], defaultdict(set)
    for number, row in enumerate(rows, start=2):
        if row["kind"] not in D1_KINDS:
            problems.append(f"line {number}: unknown kind {row['kind']!r}")
        if row["script"] != token_script(row["form"]):
            problems.append(f"line {number}: {row['form']} is not {row['script']}")
        norm = (normalise_hebrew(row["form"]) if row["script"] == "hebrew"
                else normalise_latin(row["form"])).replace(" ", "")
        groups_of_form[(row["script"], norm)].add(row["group_id"])
    problems += [f"{norm} ({script}) is in groups {sorted(groups)}"
                 for (script, norm), groups in groups_of_form.items() if len(groups) > 1]
    return problems


d1_rows = read_csv(D1_SOURCE)
_problems = d1_problems(d1_rows)
d1_forms = [{**row, "norm": (normalise_hebrew(row["form"]) if row["script"] == "hebrew"
                             else normalise_latin(row["form"])).replace(" ", ""),
             "display": display_form(row["form"], row["script"])} for row in d1_rows]
write_csv(EVAL_DIR / "d1_forms.csv", d1_forms)
REVIEW_DIR.mkdir(exist_ok=True)
write_csv(REVIEW_DIR / "d1_review.csv", [{**row, "reviewer_ok": ""} for row in d1_rows])
profile["d1"] = {"groups": len({row["group_id"] for row in d1_rows}), "forms": len(d1_rows),
                 "kinds": dict(Counter(row["kind"] for row in d1_rows))}

print(f"{RULE}\nCurated dictionary D1: {D1_SOURCE}\n{RULE}")
for _kind, _count in sorted(profile["d1"]["kinds"].items()):
    print(f"  {_kind:<13} {_count:>4} forms")
for _problem in _problems:
    print(f"  !! {_problem}")
print(f"{RULE}\n  {profile['d1']['groups']} groups, {len(d1_rows)} forms  "
      f"({'valid' if not _problems else f'{len(_problems)} problem(s)'})")

────────────────────────────────────────────────────────────────────────
Curated dictionary D1: dictionaries/d1_curated.csv
────────────────────────────────────────────────────────────────────────
  civil           26 forms
  cognate        264 forms
  hypocoristic    75 forms
  translation     56 forms
  yiddish        143 forms
────────────────────────────────────────────────────────────────────────
  105 groups, 564 forms  (valid)


**7. Consistency Check**

This cell writes `./eval/profile.json`, then reads the persons and splits files back from disk and checks them against the transcription, independently of the cells above:

1. every eligible row, recomputed from the CSV, belongs to exactly one person, and no person holds another row
2. every person has exactly one split, which `splits.csv` and its persons file agree on; development persons have a fold from 1 to 5 and test persons none
3. no family key spans two splits or two folds, so that no surname is shared between a dictionary's training folds and its evaluation fold
4. every token stored for a person re-normalises to its stored form, and the development file holds no test person, which seals the test split

It ends with the persons per split and the result of the check.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the persons and splits against the transcription
#
# Eligibility is recomputed from the raw CSV with the normalisation helpers
# only; the persons and splits are re-read from disk and compared with
# that, so an error in any earlier cell shows up here.
# ─────────────────────────────────────────────────────────────────────────────
if "d1_forms" not in globals():
    raise RuntimeError("Run every cell above first.")

(EVAL_DIR / "profile.json").write_text(json.dumps(profile, indent=1, ensure_ascii=False) + "\n",
                                       encoding="utf-8")


def stored_persons() -> dict[str, dict]:
    """Read both persons files back, keyed by person id."""
    return {row["person_id"]: row for split in ("dev", "test")
            for row in read_csv(EVAL_DIR / f"persons_{split}.csv")}


def row_problems(stored: dict[str, dict]) -> list[str]:
    """Check that the stored persons hold exactly the eligible rows, once each."""
    expected = {i for i, row in enumerate(read_csv(SOURCE_CSV))
                if is_named(row_tokens(row)) and int(row["confidence"] or 0) >= MIN_CONFIDENCE}
    owner_of_row = Counter(i for row in stored.values() for i in json.loads(row["merged_from"]))
    problems = [f"row {i}: eligible but in no person" for i in expected - set(owner_of_row)]
    problems += [f"row {i}: in a person but not eligible" for i in set(owner_of_row) - expected]
    return problems + [f"row {i}: in {n} persons" for i, n in owner_of_row.items() if n > 1]


def split_problems(stored: dict[str, dict]) -> list[str]:
    """Check splits, folds, family keys and the sealing of the development file."""
    splits = {row["person_id"]: (row["split"], row["fold"])
              for row in read_csv(EVAL_DIR / "splits.csv")}
    folds = {str(k) for k in range(1, DEV_FOLDS + 1)}
    problems, places_of_family = [], defaultdict(set)
    for person_id, row in stored.items():
        if splits.get(person_id) != (row["split"], row["fold"]):
            problems.append(f"{person_id}: split and fold disagree with splits.csv")
        if (row["split"] == "dev") != (row["fold"] in folds):
            problems.append(f"{person_id}: {row['split']} person with fold {row['fold']!r}")
        places_of_family[row["family_key"]].add((row["split"], row["fold"]))
    problems += [f"family {key}: spans {sorted(places)}"
                 for key, places in places_of_family.items() if len(places) > 1]
    problems += [f"{row['person_id']}: a test person in persons_dev.csv"
                 for row in read_csv(EVAL_DIR / "persons_dev.csv") if row["split"] != "dev"]
    return problems


def token_problems(stored: dict[str, dict]) -> list[str]:
    """Check that every stored token re-normalises to its stored form."""
    problems = []
    for person_id, row in stored.items():
        for token in json.loads(row["tokens"]):
            normalise = normalise_hebrew if token["script"] == "hebrew" else normalise_latin
            if normalise(token["display"]).replace(" ", "") != token["norm"]:
                problems.append(f"{person_id}: {token['display']} does not normalise to "
                                f"{token['norm']}")
    return problems


_stored = stored_persons()
_checks = {"rows": row_problems(_stored), "splits and folds": split_problems(_stored),
           "tokens": token_problems(_stored)}
print(f"{RULE}\nConsistency: {EVAL_DIR}/ against {SOURCE_CSV}\n{RULE}")
for _name, _problems in _checks.items():
    print(f"  {'!! ' if _problems else 'OK '}{_name:<18} "
          + (f"{len(_problems)} problem(s)" if _problems else "consistent"))
    for _problem in _problems[:10]:
        print(f"        - {_problem}")
_consistent = not any(_checks.values())
print(f"{RULE}\n  {sum(r['split'] == 'dev' for r in _stored.values())} development and "
      f"{sum(r['split'] == 'test' for r in _stored.values())} test persons  "
      f"({'consistent' if _consistent else 'inconsistent'})")
print("\n✓ Evaluation set complete (target persons, sealed splits, curated dictionary D1).")
print(f"  eval files consistent: {_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: eval/ against ../ocr_pipeline/extracted_text.csv
────────────────────────────────────────────────────────────────────────
  OK rows               consistent
  OK splits and folds   consistent
  OK tokens             consistent
────────────────────────────────────────────────────────────────────────
  507 development and 130 test persons  (consistent)

✓ Evaluation set complete (target persons, sealed splits, curated dictionary D1).
  eval files consistent: True
